# DOD-VLM End-to-End — Language-Conditioned Object Detection

> **Mục tiêu:** huấn luyện một mô hình Vision-Language cho bài toán **Described Object Detection (DOD)** từ đầu đến cuối trên RTX A3000 12 GB.  
> **Input:** ảnh RGB + một câu mô tả tự do bằng tiếng Anh.  
> **Output:** 0, 1 hoặc N bounding boxes phù hợp với toàn bộ mô tả.

Notebook này là **project độc lập**, không phụ thuộc source code cũ trong repository. Thiết kế bám theo kiến trúc đã thống nhất:

```text
Câu mô tả
   ├─► Text encoder (DistilBERT; fine-tune 2 layer cuối)
   │      ├─ full token features T
   │      ├─ target phrase embedding
   │      └─ anchor / relation embedding (nếu parse được)
   │
Ảnh
   ├─► YOLOE-26s prompt-free ─► K proposal độc lập với câu
   └─► Frozen MobileNetV3-FPN ─► MultiScale RoIAlign ─► object tokens V
                                                        │
                   ┌────────────────────────────────────┴──────────────────────┐
                   │                                                           │
                   ▼                                                           ▼
          A0 — Generic Fusion                                        A1 — Structured Fusion
   object self-attention + text cross-attn                   entity matching + pair geometry
                   │                                                           │
                   └───────────────────── learned gate ────────────────────────┘
                                             │
                                   candidate logits + null head
                                             │
                                   calibration trên validation
                                             │
                                         0 / 1 / N box
```

### Nguyên tắc vận hành

- **Run All mặc định là full training**, không phải smoke test.
- Các bước nặng được **cache/resume**; chạy lại notebook không làm lại việc đã hoàn thành.
- Proposal và ROI feature được tính **một lần trên mỗi ảnh**, sau đó YOLOE/backbone được giải phóng VRAM trước khi train VLM.
- Mỗi epoch lưu `last.pt`; metric tốt nhất lưu `best.pt`.
- Dữ liệu train có thể dùng **GT injection** khi proposal miss target; validation/test tuyệt đối không dùng GT injection.
- A0 luôn dùng toàn bộ câu, nên nếu parser A1 thất bại hệ thống vẫn có đường suy luận.

## Session 0 — Cài đặt thư viện và cấu hình chạy

Cell đầu tiên chỉ cài các package còn thiếu. PyTorch không bị cài lại nếu môi trường đã có sẵn.
Notebook yêu cầu CUDA để chạy full training hợp lý trên RTX A3000; nếu CUDA không khả dụng notebook sẽ dừng sớm với thông báo rõ ràng.

> **Lưu ý:** full gRefCOCO cần tải annotation (~180 MB), các ảnh COCO được dùng, checkpoint YOLOE và các model pretrained.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.2",
    "PIL": "Pillow>=10.0",
    "requests": "requests>=2.32",
    "tqdm": "tqdm>=4.66",
    "yaml": "PyYAML>=6.0",
    "huggingface_hub": "huggingface-hub>=0.35",
    "transformers": "transformers>=4.46,<6",
    "ultralytics": "ultralytics>=8.4.0",
    "matplotlib": "matplotlib>=3.9",
}

missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Đang cài package còn thiếu:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    print("Chưa có PyTorch/TorchVision. Đang cài bản mặc định từ PyPI...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-U", "torch>=2.3", "torchvision>=0.18"]
    )

print("✓ Hoàn tất kiểm tra/cài đặt dependency.")

In [ ]:
from __future__ import annotations

import contextlib
import dataclasses
from dataclasses import dataclass, asdict
from datetime import datetime
from functools import lru_cache
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import time
from typing import Any, Iterable

import numpy as np
import pandas as pd
from PIL import Image
import requests
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import MultiScaleRoIAlign, box_iou
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
)

from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer
from ultralytics import YOLOE

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
else:
    raise RuntimeError(
        "Full Run All cần CUDA. Hãy cài PyTorch CUDA phù hợp với NVIDIA driver rồi chạy lại."
    )

In [ ]:
@dataclass
class Config:
    # ---- Project paths ----
    project_root: str = "./dod_vlm_workspace"
    run_name: str = "dod_vlm_grefcoco_v1"

    # ---- Dataset ----
    hf_dataset_repo: str = "FudanCVL/gRefCOCO"
    split_file: str = "grefs(unc).json"
    instances_file: str = "instances.json"
    # None = dùng toàn bộ split. Có thể đặt số nhỏ hơn để debug thủ công.
    max_train_samples: int | None = None
    max_val_samples: int | None = None
    max_test_samples: int | None = None

    # ---- Proposal generator ----
    proposal_model: str = "yoloe-26s-seg-pf.pt"
    proposal_imgsz: int = 640
    proposal_conf: float = 0.03
    top_k: int = 64

    # ---- Visual object encoder ----
    roi_output_size: int = 3
    visual_dim: int = 256

    # ---- Text encoder ----
    text_model: str = "distilbert-base-uncased"
    max_text_len: int = 48
    unfreeze_last_text_layers: int = 2

    # ---- DOD core ----
    d_model: int = 256
    num_heads: int = 8
    object_layers: int = 2
    cross_layers: int = 2
    ff_dim: int = 768
    dropout: float = 0.10
    relation_geom_dim: int = 12

    # ---- Training ----
    epochs: int = 12
    batch_size: int = 12
    grad_accum_steps: int = 2
    lr_fusion: float = 3e-4
    lr_text: float = 2e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    num_workers: int = 0  # ổn định nhất cho Windows + Jupyter
    amp: bool = True
    early_stopping_patience: int = 4

    # ---- Label / GT injection ----
    positive_iou: float = 0.50
    gt_injection: bool = True
    gt_injection_prob: float = 0.70

    # ---- Loss ----
    focal_gamma: float = 2.0
    candidate_loss_weight: float = 1.0
    null_loss_weight: float = 0.50
    target_entity_aux_weight: float = 0.20

    # ---- Calibration / inference ----
    threshold_grid_size: int = 31
    default_candidate_threshold: float = 0.50
    default_null_threshold: float = 0.50

    # ---- Runtime ----
    request_timeout: int = 60
    save_every_minutes: int = 15

cfg = Config()

ROOT = Path(cfg.project_root).resolve()
DATA_DIR = ROOT / "data"
ANNOT_DIR = DATA_DIR / "grefcoco"
IMAGE_DIR = DATA_DIR / "images" / "train2014"
CACHE_DIR = ROOT / "cache"
FEATURE_DIR = CACHE_DIR / "image_features"
RUN_DIR = ROOT / "runs" / cfg.run_name
CKPT_DIR = RUN_DIR / "checkpoints"
ARTIFACT_DIR = RUN_DIR / "final_model"

for p in [ANNOT_DIR, IMAGE_DIR, FEATURE_DIR, CKPT_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG_HASH = hashlib.sha256(
    json.dumps(asdict(cfg), sort_keys=True, default=str).encode("utf-8")
).hexdigest()[:12]

print("Project root:", ROOT)
print("Config hash :", CONFIG_HASH)

## Session 1 — Hệ thống theo dõi tiến độ, timing và resume

Mỗi session ghi trạng thái vào `run_state.json`. Các file nặng được ghi theo kiểu atomic (`.tmp` → rename) để tránh cache hỏng khi mất điện/kernel crash.

- `StageTimer`: hiển thị thời gian bắt đầu/kết thúc.
- `RunState`: lưu stage đã hoàn thành.
- `atomic_torch_save` / `atomic_json_dump`: tránh checkpoint nửa chừng.
- `GPU memory`: log VRAM đang dùng sau các stage lớn.

In [ ]:
STATE_PATH = RUN_DIR / "run_state.json"

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")

def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)

def atomic_torch_save(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)

class RunState:
    def __init__(self, path: Path):
        self.path = path
        if path.exists():
            self.data = json.loads(path.read_text(encoding="utf-8"))
        else:
            self.data = {"config_hash": CONFIG_HASH, "stages": {}, "created_at": now_iso()}
        if self.data.get("config_hash") != CONFIG_HASH:
            print(
                "⚠ Config hiện tại khác run_state cũ. Cache có signature riêng nên vẫn an toàn; "
                "checkpoint train cũ sẽ chỉ resume nếu hash trùng."
            )

    def mark(self, stage: str, **meta: Any) -> None:
        self.data["config_hash"] = CONFIG_HASH
        self.data["stages"][stage] = {"completed_at": now_iso(), **meta}
        atomic_json_dump(self.data, self.path)

    def done(self, stage: str) -> bool:
        return stage in self.data.get("stages", {})

state = RunState(STATE_PATH)

@contextlib.contextmanager
def stage_timer(name: str):
    start = time.perf_counter()
    print(f"\n{'='*88}\n▶ {name}\nBắt đầu: {now_iso()}\n{'='*88}")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"✓ {name} — {elapsed/60:.2f} phút")

def gpu_report(prefix: str = "") -> None:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 2**30
        reserv = torch.cuda.memory_reserved() / 2**30
        peak = torch.cuda.max_memory_allocated() / 2**30
        print(f"{prefix} GPU memory: allocated={alloc:.2f} GB | reserved={reserv:.2f} GB | peak={peak:.2f} GB")

## Session 2 — Tải và chuẩn hóa gRefCOCO

gRefCOCO được dùng vì nó hỗ trợ **0/1/N target**. Notebook tải trực tiếp annotation chính thức từ Hugging Face, sau đó chuyển thành schema nội bộ thống nhất:

```python
{
    "sample_id": ...,
    "split": "train|val|test...",
    "image_id": ...,
    "file_name": ...,
    "query": ...,
    "gt_boxes": [[x1,y1,x2,y2], ...],
    "target_categories": [...],
    "no_target": bool,
}
```

`ann_id=-1` / annotation không tồn tại được hiểu là no-target. Multi-target giữ toàn bộ GT boxes.

In [ ]:
def download_annotations() -> tuple[Path, Path]:
    with stage_timer("Session 2A — Download annotation gRefCOCO"):
        local = Path(
            snapshot_download(
                repo_id=cfg.hf_dataset_repo,
                repo_type="dataset",
                allow_patterns=[cfg.split_file, cfg.instances_file],
                local_dir=ANNOT_DIR,
            )
        )
        refs_path = local / cfg.split_file
        instances_path = local / cfg.instances_file
        if not refs_path.exists() or not instances_path.exists():
            raise FileNotFoundError("Không tìm thấy annotation gRefCOCO sau khi download.")
        print("Refs     :", refs_path)
        print("Instances:", instances_path)
        state.mark("annotations_downloaded", refs=str(refs_path), instances=str(instances_path))
        return refs_path, instances_path

REFS_PATH, INSTANCES_PATH = download_annotations()

In [ ]:
MANIFEST_PATH = DATA_DIR / f"manifest_{CONFIG_HASH}.jsonl"
IMAGE_META_PATH = DATA_DIR / f"image_meta_{CONFIG_HASH}.json"

def xywh_to_xyxy(box: Iterable[float]) -> list[float]:
    x, y, w, h = map(float, box)
    return [x, y, x + w, y + h]

def build_manifest(refs_path: Path, instances_path: Path) -> pd.DataFrame:
    if MANIFEST_PATH.exists() and IMAGE_META_PATH.exists():
        print("✓ Dùng lại manifest đã chuẩn hóa:", MANIFEST_PATH)
        return pd.read_json(MANIFEST_PATH, lines=True)

    with stage_timer("Session 2B — Chuẩn hóa annotation thành DOD manifest"):
        refs = json.loads(refs_path.read_text(encoding="utf-8"))
        instances = json.loads(instances_path.read_text(encoding="utf-8"))

        anns = {int(a["id"]): a for a in instances["annotations"]}
        images = {int(x["id"]): x for x in instances["images"]}
        cats = {int(c["id"]): c["name"] for c in instances["categories"]}

        rows: list[dict[str, Any]] = []
        for ref in tqdm(refs, desc="Chuẩn hóa refs", unit="ref"):
            image_id = int(ref["image_id"])
            image_meta = images[image_id]

            ann_ids = ref.get("ann_id", [])
            if not isinstance(ann_ids, list):
                ann_ids = [ann_ids]

            valid_anns = [anns[int(a)] for a in ann_ids if int(a) in anns and int(a) != -1]
            gt_boxes = [xywh_to_xyxy(a["bbox"]) for a in valid_anns]

            category_ids = ref.get("category_id", [])
            if not isinstance(category_ids, list):
                category_ids = [category_ids]
            target_categories = [cats.get(int(c), str(c)) for c in category_ids if int(c) in cats]

            for sent in ref.get("sentences", []):
                query = str(sent.get("sent", "")).strip()
                if not query:
                    continue
                rows.append(
                    {
                        "sample_id": f'{ref["ref_id"]}_{sent["sent_id"]}',
                        "ref_id": int(ref["ref_id"]),
                        "sent_id": int(sent["sent_id"]),
                        "split": str(ref["split"]),
                        "image_id": image_id,
                        "file_name": image_meta["file_name"],
                        "coco_url": image_meta.get(
                            "coco_url",
                            f'http://images.cocodataset.org/train2014/{image_meta["file_name"]}',
                        ),
                        "width": int(image_meta["width"]),
                        "height": int(image_meta["height"]),
                        "query": query,
                        "gt_boxes": gt_boxes,
                        "target_categories": target_categories,
                        "no_target": len(gt_boxes) == 0,
                    }
                )

        df = pd.DataFrame(rows)
        if df.empty:
            raise RuntimeError("Manifest rỗng — schema gRefCOCO có thể đã thay đổi.")

        # Limit chỉ dùng nếu người dùng chủ động cấu hình.
        limits = {
            "train": cfg.max_train_samples,
            "val": cfg.max_val_samples,
            "test": cfg.max_test_samples,
        }
        parts = []
        for split, part in df.groupby("split", sort=False):
            limit = limits.get(split)
            parts.append(part.iloc[:limit] if limit else part)
        df = pd.concat(parts, ignore_index=True)

        df.to_json(MANIFEST_PATH, orient="records", lines=True, force_ascii=False)
        image_meta = (
            df[["image_id", "file_name", "coco_url", "width", "height"]]
            .drop_duplicates("image_id")
            .to_dict("records")
        )
        atomic_json_dump(image_meta, IMAGE_META_PATH)

        print(df.groupby("split").agg(samples=("sample_id", "size"), no_target=("no_target", "sum")))
        print("Unique images:", df["image_id"].nunique())
        state.mark("manifest_built", samples=len(df), images=int(df["image_id"].nunique()))
        return df

manifest = build_manifest(REFS_PATH, INSTANCES_PATH)
display(manifest.head(3))

## Session 3 — Tải ảnh COCO cần thiết

Notebook **không tải cả COCO train2014 một cách mù quáng**. Nó chỉ tải các `image_id` thực sự xuất hiện trong manifest hiện tại.

Mỗi ảnh:
- nếu đã tồn tại và mở được → skip;
- nếu download lỗi → retry;
- file được ghi `.part` rồi rename để tránh ảnh dở dang.

In [ ]:
def download_one_image(row: dict[str, Any], retries: int = 4) -> None:
    out = IMAGE_DIR / row["file_name"]
    if out.exists():
        try:
            with Image.open(out) as im:
                im.verify()
            return
        except Exception:
            out.unlink(missing_ok=True)

    urls = [
        row.get("coco_url"),
        f'http://images.cocodataset.org/train2014/{row["file_name"]}',
        f'https://images.cocodataset.org/train2014/{row["file_name"]}',
    ]
    urls = [u for u in dict.fromkeys(urls) if u]
    part = out.with_suffix(out.suffix + ".part")

    last_err: Exception | None = None
    for attempt in range(retries):
        for url in urls:
            try:
                with requests.get(url, stream=True, timeout=cfg.request_timeout) as r:
                    r.raise_for_status()
                    with open(part, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1 << 20):
                            if chunk:
                                f.write(chunk)
                with Image.open(part) as im:
                    im.verify()
                part.replace(out)
                return
            except Exception as exc:
                last_err = exc
                part.unlink(missing_ok=True)
        time.sleep(min(2**attempt, 8))
    raise RuntimeError(f"Không tải được {row['file_name']}: {last_err}")

with stage_timer("Session 3 — Download COCO images"):
    image_rows = json.loads(IMAGE_META_PATH.read_text(encoding="utf-8"))
    for row in tqdm(image_rows, desc="COCO images", unit="img"):
        download_one_image(row)
    state.mark("images_downloaded", count=len(image_rows))

## Session 4 — Cache prompt-independent proposals và object tokens

Đây là bước tốn thời gian nhất nhưng **chỉ chạy một lần cho mỗi ảnh**.

### 4.1 Proposal
`YOLOE-26s-seg-pf.pt` sinh proposal với vocabulary tích hợp, hoàn toàn **không phụ thuộc câu lệnh**.

### 4.2 Object visual token
Để không phụ thuộc embedding nội bộ/undocumented của YOLOE, notebook dùng một visual encoder độc lập:

- pretrained `FasterRCNN-MobileNetV3-Large-FPN`;
- lấy frozen FPN features;
- `MultiScaleRoIAlign` trên box YOLOE;
- average pool thành vector 256 chiều cho từng candidate.

Cache mỗi ảnh chứa:
- boxes `[K,4]`,
- detector scores `[K]`,
- detector class names,
- ROI features `[K,256]`.

### 4.3 GT injection cho train
ROI feature của toàn bộ GT box xuất hiện trong **train split** cũng được cache riêng. Chúng chỉ được chèn xác suất `gt_injection_prob` nếu proposal không cover target. Validation/test không bao giờ dùng GT injection.

In [ ]:
CACHE_SIGNATURE = hashlib.sha256(
    json.dumps(
        {
            "proposal_model": cfg.proposal_model,
            "imgsz": cfg.proposal_imgsz,
            "conf": cfg.proposal_conf,
            "top_k": cfg.top_k,
            "roi_output": cfg.roi_output_size,
            "visual_dim": cfg.visual_dim,
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:12]
FEATURE_RUN_DIR = FEATURE_DIR / CACHE_SIGNATURE
FEATURE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Feature cache signature:", CACHE_SIGNATURE)

class FrozenRegionEncoder:
    def __init__(self, device: torch.device):
        weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
        model = fasterrcnn_mobilenet_v3_large_fpn(weights=weights)
        self.backbone = model.backbone.to(device).eval()
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.roi = MultiScaleRoIAlign(
            featmap_names=["0", "1", "pool"],
            output_size=cfg.roi_output_size,
            sampling_ratio=2,
        )
        self.device = device
        self.transform = weights.transforms()

    @torch.inference_mode()
    def encode(self, image_pil: Image.Image, boxes: torch.Tensor) -> torch.Tensor:
        if boxes.numel() == 0:
            return torch.zeros((0, cfg.visual_dim), dtype=torch.float32)

        image = self.transform(image_pil).to(self.device)
        _, h, w = image.shape
        feats = self.backbone(image.unsqueeze(0))
        pooled = self.roi(
            feats,
            [boxes.to(self.device)],
            [(h, w)],
        )
        pooled = pooled.mean(dim=(-1, -2))
        return pooled.float().cpu()

def collect_train_gt_by_image(df: pd.DataFrame) -> dict[int, torch.Tensor]:
    train = df[df["split"] == "train"]
    out: dict[int, list[list[float]]] = {}
    for row in train.itertuples():
        out.setdefault(int(row.image_id), []).extend(row.gt_boxes)
    unique = {}
    for image_id, boxes in out.items():
        if boxes:
            arr = torch.tensor(boxes, dtype=torch.float32)
            # simple exact-ish dedup
            rounded = torch.round(arr * 10) / 10
            _, idx = torch.unique(rounded, dim=0, return_inverse=False, return_counts=False), None
            # preserve order robustly
            seen, keep = set(), []
            for i, b in enumerate(rounded.tolist()):
                key = tuple(b)
                if key not in seen:
                    seen.add(key)
                    keep.append(i)
            unique[image_id] = arr[keep]
        else:
            unique[image_id] = torch.zeros((0, 4), dtype=torch.float32)
    return unique

train_gt_by_image = collect_train_gt_by_image(manifest)

In [ ]:
def cache_path_for_image(image_id: int) -> Path:
    return FEATURE_RUN_DIR / f"{int(image_id):012d}.pt"

def cache_is_valid(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
        return obj.get("cache_signature") == CACHE_SIGNATURE
    except Exception:
        return False

with stage_timer("Session 4 — Cache YOLOE proposals + ROI object features"):
    proposal = YOLOE(cfg.proposal_model)
    region_encoder = FrozenRegionEncoder(DEVICE)

    unique_images = (
        manifest[["image_id", "file_name", "width", "height"]]
        .drop_duplicates("image_id")
        .to_dict("records")
    )

    completed = sum(cache_is_valid(cache_path_for_image(x["image_id"])) for x in unique_images)
    pbar = tqdm(unique_images, desc=f"Feature cache ({completed}/{len(unique_images)} sẵn)", unit="img")

    for item in pbar:
        image_id = int(item["image_id"])
        out_path = cache_path_for_image(image_id)
        if cache_is_valid(out_path):
            continue

        image_path = IMAGE_DIR / item["file_name"]
        image_pil = Image.open(image_path).convert("RGB")

        result = proposal.predict(
            source=str(image_path),
            imgsz=cfg.proposal_imgsz,
            conf=cfg.proposal_conf,
            max_det=cfg.top_k,
            device=0,
            half=True,
            verbose=False,
        )[0]

        if result.boxes is None or len(result.boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            scores = torch.zeros((0,), dtype=torch.float32)
            labels: list[str] = []
        else:
            boxes = result.boxes.xyxy.detach().cpu().float()[: cfg.top_k]
            scores = result.boxes.conf.detach().cpu().float()[: cfg.top_k]
            cls_ids = result.boxes.cls.detach().cpu().long().tolist()[: cfg.top_k]
            labels = [str(result.names[int(i)]) for i in cls_ids]

        proposal_features = region_encoder.encode(image_pil, boxes)

        gt_boxes = train_gt_by_image.get(image_id, torch.zeros((0, 4), dtype=torch.float32))
        gt_features = region_encoder.encode(image_pil, gt_boxes)

        payload = {
            "cache_signature": CACHE_SIGNATURE,
            "image_id": image_id,
            "boxes": boxes,
            "scores": scores,
            "labels": labels,
            "features": proposal_features,
            "train_gt_boxes": gt_boxes,
            "train_gt_features": gt_features,
            "image_size": (int(item["height"]), int(item["width"])),
        }
        atomic_torch_save(payload, out_path)

    del proposal, region_encoder
    gc.collect()
    torch.cuda.empty_cache()
    gpu_report("Sau khi giải phóng proposal/ROI encoder:")
    state.mark("feature_cache_complete", signature=CACHE_SIGNATURE, images=len(unique_images))

## Session 5 — Kiến trúc VLM DOD

Đây là phần **được huấn luyện** của hệ thống.

### A0 — Generic multimodal fusion
1. visual projection đưa ROI feature về `d_model`;
2. geometry embedding của từng box;
3. self-attention giữa K object token;
4. cross-attention từ object token sang toàn bộ token của câu;
5. tạo `generic_logit` cho từng candidate.

### A1 — Structured relational grounding
1. target phrase và anchor phrase được mã hóa bởi cùng text encoder;
2. học `target_match(i)` và `anchor_match(j)`;
3. tính geometry cho mọi cặp `(i,j)`;
4. relation-conditioned MLP tính `R(i,j | relation)`;
5. anchor là latent variable, softmax trên `anchor_match`;
6. aggregate relation support về từng target candidate;
7. tạo `structured_logit`.

### Fusion + no-target
- Query-dependent gate học cách trộn A0/A1.
- Candidate output là sigmoid độc lập → hỗ trợ multi-target.
- Global null head học `P(no target)` riêng.

In [ ]:
RELATION_PATTERNS = [
    (r"\bto the left of\b|\bleft of\b", "left_of"),
    (r"\bto the right of\b|\bright of\b", "right_of"),
    (r"\babove\b|\bover\b", "above"),
    (r"\bbelow\b|\bunder\b", "below"),
    (r"\bnext to\b|\bbeside\b|\bnear\b", "near"),
    (r"\bin front of\b", "in_front_of"),
    (r"\bbehind\b", "behind"),
    (r"\binside\b|\bin\b", "inside"),
    (r"\bon top of\b", "on_top_of"),
    (r"\boverlapping\b", "overlap"),
]
RELATION_NAMES = ["none", "left_of", "right_of", "above", "below", "near", "in_front_of", "behind", "inside", "on_top_of", "overlap"]
REL2ID = {x: i for i, x in enumerate(RELATION_NAMES)}

def clean_phrase(x: str) -> str:
    x = re.sub(r"\b(the|a|an|please|find|locate|detect|show|me)\b", " " , x.lower())
    x = re.sub(r"\s+", " " , x).strip(" ,.;:")
    return x

def parse_query(query: str, target_categories: list[str] | None = None) -> dict[str, str | int]:
    q = query.strip()
    ql = q.lower()

    relation_name = "none"
    anchor_phrase = ""
    left = q

    for pattern, name in RELATION_PATTERNS:
        m = re.search(pattern, ql)
        if m:
            relation_name = name
            left = q[:m.start()]
            anchor_phrase = clean_phrase(q[m.end():])
            break

    if target_categories:
        target_phrase = ", ".join(target_categories)
    else:
        left_clean = clean_phrase(left)
        # heuristic fallback: giữ tối đa 5 token cuối trước relation
        toks = left_clean.split()
        target_phrase = " ".join(toks[-5:]) if toks else clean_phrase(q)

    return {
        "target_phrase": target_phrase or clean_phrase(q),
        "anchor_phrase": anchor_phrase,
        "relation_name": relation_name,
        "relation_id": REL2ID[relation_name],
    }

def normalize_boxes(boxes: torch.Tensor, image_sizes: torch.Tensor) -> torch.Tensor:
    # boxes [B,K,4], image_sizes [B,2]=(h,w)
    h = image_sizes[:, 0:1]
    w = image_sizes[:, 1:2]
    scale = torch.stack([w, h, w, h], dim=-1).squeeze(1)
    return boxes / scale[:, None, :].clamp_min(1.0)

def box_geometry(boxes_norm: torch.Tensor) -> torch.Tensor:
    x1, y1, x2, y2 = boxes_norm.unbind(-1)
    cx, cy = (x1+x2)/2, (y1+y2)/2
    bw, bh = (x2-x1).clamp_min(1e-6), (y2-y1).clamp_min(1e-6)
    area = bw * bh
    return torch.stack([cx, cy, bw, bh, area, x1, y1, x2, y2], dim=-1)

def pair_geometry(boxes_norm: torch.Tensor) -> torch.Tensor:
    # [B,K,K,12]
    x1, y1, x2, y2 = boxes_norm.unbind(-1)
    cx, cy = (x1+x2)/2, (y1+y2)/2
    w, h = (x2-x1).clamp_min(1e-6), (y2-y1).clamp_min(1e-6)

    dxc = cx[:, :, None] - cx[:, None, :]
    dyc = cy[:, :, None] - cy[:, None, :]
    log_wr = torch.log(w[:, :, None] / w[:, None, :])
    log_hr = torch.log(h[:, :, None] / h[:, None, :])

    left_margin = x1[:, None, :] - x2[:, :, None]   # positive nếu i nằm trái j
    right_margin = x1[:, :, None] - x2[:, None, :]
    above_margin = y1[:, None, :] - y2[:, :, None]
    below_margin = y1[:, :, None] - y2[:, None, :]

    inter_x1 = torch.maximum(x1[:, :, None], x1[:, None, :])
    inter_y1 = torch.maximum(y1[:, :, None], y1[:, None, :])
    inter_x2 = torch.minimum(x2[:, :, None], x2[:, None, :])
    inter_y2 = torch.minimum(y2[:, :, None], y2[:, None, :])
    inter = (inter_x2-inter_x1).clamp_min(0) * (inter_y2-inter_y1).clamp_min(0)
    area_i = w[:, :, None] * h[:, :, None]
    area_j = w[:, None, :] * h[:, None, :]
    iou = inter / (area_i + area_j - inter).clamp_min(1e-6)

    dist = torch.sqrt(dxc.square() + dyc.square() + 1e-8)
    overlap_x = (inter_x2-inter_x1).clamp_min(0)
    overlap_y = (inter_y2-inter_y1).clamp_min(0)

    return torch.stack(
        [
            dxc, dyc, log_wr, log_hr,
            left_margin, right_margin, above_margin, below_margin,
            iou, dist, overlap_x, overlap_y,
        ],
        dim=-1,
    )

In [ ]:
class DODVLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        self.text_encoder = AutoModel.from_pretrained(cfg.text_model)
        text_dim = int(self.text_encoder.config.hidden_size)

        # Freeze toàn bộ rồi mở N layer cuối.
        for p in self.text_encoder.parameters():
            p.requires_grad_(False)
        layers = getattr(self.text_encoder, "transformer").layer
        for layer in layers[-cfg.unfreeze_last_text_layers:]:
            for p in layer.parameters():
                p.requires_grad_(True)

        self.visual_proj = nn.Sequential(
            nn.Linear(cfg.visual_dim, cfg.d_model),
            nn.LayerNorm(cfg.d_model),
            nn.GELU(),
        )
        self.geom_proj = nn.Sequential(
            nn.Linear(9, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.d_model),
        )
        self.text_proj = nn.Linear(text_dim, cfg.d_model)

        obj_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.num_heads,
            dim_feedforward=cfg.ff_dim,
            dropout=cfg.dropout,
            batch_first=True,
            norm_first=True,
        )
        self.object_encoder = nn.TransformerEncoder(obj_layer, num_layers=cfg.object_layers)

        self.cross_attn = nn.ModuleList(
            [
                nn.MultiheadAttention(
                    embed_dim=cfg.d_model,
                    num_heads=cfg.num_heads,
                    dropout=cfg.dropout,
                    batch_first=True,
                )
                for _ in range(cfg.cross_layers)
            ]
        )
        self.cross_norm = nn.ModuleList([nn.LayerNorm(cfg.d_model) for _ in range(cfg.cross_layers)])
        self.generic_head = nn.Sequential(
            nn.Linear(cfg.d_model * 2 + 1, cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, 1),
        )

        # A1 entity matching
        self.entity_visual = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.entity_text = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

        self.relation_embed = nn.Embedding(len(RELATION_NAMES), cfg.d_model)
        self.pair_geom_mlp = nn.Sequential(
            nn.Linear(cfg.relation_geom_dim, cfg.d_model // 2),
            nn.GELU(),
            nn.Linear(cfg.d_model // 2, cfg.d_model // 2),
        )
        self.relation_mlp = nn.Sequential(
            nn.Linear(cfg.d_model * 3 + cfg.d_model // 2, cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, 1),
        )
        self.structured_head = nn.Sequential(
            nn.Linear(cfg.d_model + 3, cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, 1),
        )

        self.fusion_gate = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_model // 2),
            nn.GELU(),
            nn.Linear(cfg.d_model // 2, 1),
        )

        self.null_head = nn.Sequential(
            nn.Linear(cfg.d_model * 2 + 2, cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, 1),
        )

    def encode_text(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        tokens = self.text_proj(out.last_hidden_state)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (tokens * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return tokens, pooled

    def encode_phrase(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        tokens, pooled = self.encode_text(input_ids, attention_mask)
        return pooled

    def forward(
        self,
        visual_features: torch.Tensor,      # [B,K,256]
        boxes: torch.Tensor,                # [B,K,4]
        detector_scores: torch.Tensor,      # [B,K]
        candidate_mask: torch.Tensor,       # [B,K]
        image_sizes: torch.Tensor,          # [B,2]
        query_ids: torch.Tensor,
        query_mask: torch.Tensor,
        target_ids: torch.Tensor,
        target_text_mask: torch.Tensor,
        anchor_ids: torch.Tensor,
        anchor_text_mask: torch.Tensor,
        relation_ids: torch.Tensor,
        has_relation: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        bsz, k, _ = visual_features.shape

        query_tokens, query_pooled = self.encode_text(query_ids, query_mask)
        target_pooled = self.encode_phrase(target_ids, target_text_mask)
        anchor_pooled = self.encode_phrase(anchor_ids, anchor_text_mask)

        boxes_norm = normalize_boxes(boxes, image_sizes)
        x = self.visual_proj(visual_features) + self.geom_proj(box_geometry(boxes_norm))
        x = self.object_encoder(x, src_key_padding_mask=~candidate_mask)

        # ---------- A0 generic ----------
        a0 = x
        text_pad = ~query_mask.bool()
        for attn, norm in zip(self.cross_attn, self.cross_norm):
            delta, _ = attn(a0, query_tokens, query_tokens, key_padding_mask=text_pad, need_weights=False)
            a0 = norm(a0 + delta)

        q = query_pooled[:, None, :].expand(-1, k, -1)
        generic_logits = self.generic_head(
            torch.cat([a0, q, detector_scores.unsqueeze(-1)], dim=-1)
        ).squeeze(-1)

        # ---------- A1 structured ----------
        v_ent = F.normalize(self.entity_visual(x), dim=-1)
        t_ent = F.normalize(self.entity_text(target_pooled), dim=-1)
        a_ent = F.normalize(self.entity_text(anchor_pooled), dim=-1)

        target_match = (v_ent * t_ent[:, None, :]).sum(-1)
        anchor_match = (v_ent * a_ent[:, None, :]).sum(-1)

        pair_g = self.pair_geom_mlp(pair_geometry(boxes_norm))
        vi = x[:, :, None, :].expand(-1, k, k, -1)
        vj = x[:, None, :, :].expand(-1, k, k, -1)
        r = self.relation_embed(relation_ids)[:, None, None, :].expand(-1, k, k, -1)

        relation_logits = self.relation_mlp(torch.cat([vi, vj, r, pair_g], dim=-1)).squeeze(-1)

        pair_valid = candidate_mask[:, :, None] & candidate_mask[:, None, :]
        eye = torch.eye(k, dtype=torch.bool, device=x.device)[None]
        pair_valid = pair_valid & ~eye

        anchor_scores = anchor_match.masked_fill(~candidate_mask, -1e4)
        anchor_weights = torch.softmax(anchor_scores, dim=-1)
        rel_support = (relation_logits.masked_fill(~pair_valid, -1e4).softmax(-1) * anchor_weights[:, None, :]).sum(-1)
        rel_support = torch.where(has_relation[:, None], rel_support, torch.zeros_like(rel_support))

        structured_logits = self.structured_head(
            torch.cat(
                [
                    x,
                    detector_scores.unsqueeze(-1),
                    target_match.unsqueeze(-1),
                    rel_support.unsqueeze(-1),
                ],
                dim=-1,
            )
        ).squeeze(-1)

        gate = torch.sigmoid(self.fusion_gate(query_pooled))
        candidate_logits = (1.0 - gate) * generic_logits + gate * structured_logits
        candidate_logits = candidate_logits.masked_fill(~candidate_mask, -1e4)

        maskf = candidate_mask.float().unsqueeze(-1)
        pooled_visual = (x * maskf).sum(1) / maskf.sum(1).clamp_min(1.0)
        best_logit = candidate_logits.masked_fill(~candidate_mask, -1e4).max(1).values.unsqueeze(-1)
        mean_score = (
            detector_scores.masked_fill(~candidate_mask, 0.0).sum(1)
            / candidate_mask.sum(1).clamp_min(1)
        ).unsqueeze(-1)

        null_logit = self.null_head(
            torch.cat([pooled_visual, query_pooled, best_logit, mean_score], dim=-1)
        ).squeeze(-1)

        return {
            "candidate_logits": candidate_logits,
            "null_logit": null_logit,
            "generic_logits": generic_logits,
            "structured_logits": structured_logits,
            "target_match": target_match,
            "anchor_match": anchor_match,
            "relation_logits": relation_logits,
            "fusion_gate": gate.squeeze(-1),
        }

## Session 6 — Dataset loader, candidate labels và batching

Mỗi sample đọc cache của **ảnh** rồi tạo target cho **câu**:

- candidate positive nếu IoU với bất kỳ GT ≥ `positive_iou`;
- no-target → toàn bộ candidate label = 0, `null=1`;
- nếu train sample có target nhưng proposal miss, notebook có thể chèn GT candidate đã cache;
- padding về `top_k`;
- tokenizer chạy trong collate để text encoder có thể fine-tune.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.text_model)

def candidate_labels(boxes: torch.Tensor, gt_boxes: torch.Tensor, threshold: float) -> torch.Tensor:
    if boxes.numel() == 0 or gt_boxes.numel() == 0:
        return torch.zeros((boxes.shape[0],), dtype=torch.float32)
    ious = box_iou(boxes, gt_boxes)
    return (ious.max(dim=1).values >= threshold).float()

class CachedDODDataset(Dataset):
    def __init__(self, df: pd.DataFrame, split: str, training: bool):
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.split = split
        self.training = training

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.df.iloc[idx]
        cache = torch.load(
            cache_path_for_image(int(row.image_id)),
            map_location="cpu",
            weights_only=False,
        )

        boxes = cache["boxes"].clone()
        scores = cache["scores"].clone()
        features = cache["features"].clone()
        labels = list(cache["labels"])

        gt_boxes = torch.tensor(row.gt_boxes, dtype=torch.float32).reshape(-1, 4)
        y = candidate_labels(boxes, gt_boxes, cfg.positive_iou)

        injected = False
        if (
            self.training
            and cfg.gt_injection
            and gt_boxes.numel() > 0
            and y.sum() == 0
            and random.random() < cfg.gt_injection_prob
        ):
            train_gt_boxes = cache["train_gt_boxes"]
            train_gt_features = cache["train_gt_features"]
            if train_gt_boxes.numel() > 0:
                iou = box_iou(train_gt_boxes, gt_boxes)
                keep_gt = torch.where(iou.max(dim=1).values >= 0.95)[0]
                if len(keep_gt) > 0:
                    add_boxes = train_gt_boxes[keep_gt]
                    add_feats = train_gt_features[keep_gt]
                    boxes = torch.cat([boxes, add_boxes], dim=0)
                    features = torch.cat([features, add_feats], dim=0)
                    scores = torch.cat([scores, torch.zeros(len(keep_gt))], dim=0)
                    labels.extend(["__gt_injected__"] * len(keep_gt))
                    y = candidate_labels(boxes, gt_boxes, cfg.positive_iou)
                    injected = True

        # ưu tiên positive, sau đó detector score
        if len(boxes) > cfg.top_k:
            priority = y * 10.0 + scores
            keep = torch.topk(priority, k=cfg.top_k).indices
            boxes, scores, features, y = boxes[keep], scores[keep], features[keep], y[keep]
            labels = [labels[int(i)] for i in keep.tolist()]

        parsed = parse_query(str(row.query), list(row.target_categories))
        return {
            "sample_id": row.sample_id,
            "query": str(row.query),
            "target_phrase": parsed["target_phrase"],
            "anchor_phrase": parsed["anchor_phrase"] or "[UNK]",
            "relation_id": int(parsed["relation_id"]),
            "has_relation": int(parsed["relation_id"]) != 0,
            "boxes": boxes,
            "scores": scores,
            "features": features,
            "candidate_y": y,
            "gt_boxes": gt_boxes,
            "null_y": float(bool(row.no_target)),
            "image_size": torch.tensor([row.height, row.width], dtype=torch.float32),
            "injected": injected,
        }

def pad_1d(x: torch.Tensor, length: int, value: float = 0.0) -> torch.Tensor:
    out = x.new_full((length,), value)
    out[: min(length, len(x))] = x[:length]
    return out

def pad_2d(x: torch.Tensor, length: int, width: int, value: float = 0.0) -> torch.Tensor:
    out = x.new_full((length, width), value)
    n = min(length, x.shape[0])
    out[:n] = x[:n]
    return out

def collate_dod(batch: list[dict[str, Any]]) -> dict[str, Any]:
    k = cfg.top_k
    query_tok = tokenizer(
        [x["query"] for x in batch],
        padding=True,
        truncation=True,
        max_length=cfg.max_text_len,
        return_tensors="pt",
    )
    target_tok = tokenizer(
        [x["target_phrase"] for x in batch],
        padding=True,
        truncation=True,
        max_length=16,
        return_tensors="pt",
    )
    anchor_tok = tokenizer(
        [x["anchor_phrase"] for x in batch],
        padding=True,
        truncation=True,
        max_length=16,
        return_tensors="pt",
    )

    counts = [min(k, len(x["boxes"])) for x in batch]
    mask = torch.zeros((len(batch), k), dtype=torch.bool)
    for i, n in enumerate(counts):
        mask[i, :n] = True

    return {
        "sample_id": [x["sample_id"] for x in batch],
        "query": [x["query"] for x in batch],
        "visual_features": torch.stack([pad_2d(x["features"], k, cfg.visual_dim) for x in batch]),
        "boxes": torch.stack([pad_2d(x["boxes"], k, 4) for x in batch]),
        "detector_scores": torch.stack([pad_1d(x["scores"], k) for x in batch]),
        "candidate_mask": mask,
        "candidate_y": torch.stack([pad_1d(x["candidate_y"], k) for x in batch]),
        "null_y": torch.tensor([x["null_y"] for x in batch], dtype=torch.float32),
        "image_sizes": torch.stack([x["image_size"] for x in batch]),
        "gt_boxes": [x["gt_boxes"] for x in batch],
        "query_ids": query_tok["input_ids"],
        "query_mask": query_tok["attention_mask"],
        "target_ids": target_tok["input_ids"],
        "target_text_mask": target_tok["attention_mask"],
        "anchor_ids": anchor_tok["input_ids"],
        "anchor_text_mask": anchor_tok["attention_mask"],
        "relation_ids": torch.tensor([x["relation_id"] for x in batch], dtype=torch.long),
        "has_relation": torch.tensor([x["has_relation"] for x in batch], dtype=torch.bool),
        "injected": torch.tensor([x["injected"] for x in batch], dtype=torch.bool),
    }

def make_loader(
    split: str,
    training: bool,
    shuffle: bool,
    seed: int | None = None,
) -> DataLoader:
    ds = CachedDODDataset(manifest, split=split, training=training)
    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=cfg.num_workers,
        collate_fn=collate_dod,
        pin_memory=True,
        drop_last=training,
    )

val_loader = make_loader("val", training=False, shuffle=False)

test_split = "test" if "test" in set(manifest["split"]) else (
    "testA" if "testA" in set(manifest["split"]) else "val"
)
test_loader = make_loader(test_split, training=False, shuffle=False)

train_samples = int((manifest["split"] == "train").sum())
train_batches = train_samples // cfg.batch_size
print("Train samples:", train_samples, "| ~batches/epoch:", train_batches)
print("Val batches   :", len(val_loader))
print("Test split    :", test_split, "| batches:", len(test_loader))

## Session 7 — Loss, metric, checkpoint và training có resume

### Loss
- **Candidate focal BCE:** học 0/1/N candidate.
- **Null BCE:** học explicit no-target.
- **Target entity auxiliary loss:** ép structured branch giữ target phrase alignment.

### Checkpoint
- `recovery.pt`: ghi định kỳ trong epoch theo `save_every_minutes` để tiếp tục gần đúng batch bị dừng;
- `last.pt`: ghi sau mỗi epoch;
- `best.pt`: metric validation tốt nhất;
- optimizer, scheduler, GradScaler, epoch, best metric và config hash đều được lưu;
- chạy lại notebook sẽ tự resume nếu `last.pt` cùng config hash.

### Metric chính
`set_f1@0.5` được tính trên tập box output sau threshold:
- matching greedy theo IoU ≥ 0.5;
- hỗ trợ 0/1/N;
- đồng thời báo `no_target_accuracy` và `false_boxes_per_negative`.

In [ ]:
def move_batch(batch: dict[str, Any]) -> dict[str, Any]:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(DEVICE, non_blocking=True)
        else:
            out[k] = v
    return out

def focal_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
    gamma: float,
) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    pt = p * targets + (1 - p) * (1 - targets)
    loss = ((1 - pt).pow(gamma) * bce) * mask.float()
    return loss.sum() / mask.float().sum().clamp_min(1.0)

def compute_loss(out: dict[str, torch.Tensor], batch: dict[str, Any]) -> tuple[torch.Tensor, dict[str, float]]:
    cand = focal_bce_with_logits(
        out["candidate_logits"],
        batch["candidate_y"],
        batch["candidate_mask"],
        cfg.focal_gamma,
    )
    null = F.binary_cross_entropy_with_logits(out["null_logit"], batch["null_y"])

    pos_mask = (batch["candidate_y"] > 0.5) & batch["candidate_mask"]
    if pos_mask.any():
        aux = F.binary_cross_entropy_with_logits(
            out["target_match"][pos_mask],
            torch.ones_like(out["target_match"][pos_mask]),
        )
    else:
        aux = out["target_match"].sum() * 0.0

    total = (
        cfg.candidate_loss_weight * cand
        + cfg.null_loss_weight * null
        + cfg.target_entity_aux_weight * aux
    )
    return total, {"candidate": float(cand.detach()), "null": float(null.detach()), "target_aux": float(aux.detach())}

def greedy_match_count(pred: torch.Tensor, gt: torch.Tensor, iou_thr: float = 0.5) -> tuple[int, int, int]:
    if len(pred) == 0 and len(gt) == 0:
        return 0, 0, 0
    if len(pred) == 0:
        return 0, 0, len(gt)
    if len(gt) == 0:
        return 0, len(pred), 0

    ious = box_iou(pred.cpu(), gt.cpu())
    tp = 0
    used_p, used_g = set(), set()
    while True:
        val = -1.0
        pair = None
        for i in range(ious.shape[0]):
            if i in used_p:
                continue
            for j in range(ious.shape[1]):
                if j in used_g:
                    continue
                if float(ious[i, j]) > val:
                    val = float(ious[i, j])
                    pair = (i, j)
        if pair is None or val < iou_thr:
            break
        used_p.add(pair[0]); used_g.add(pair[1]); tp += 1
    return tp, len(pred) - tp, len(gt) - tp

@torch.inference_mode()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    candidate_threshold: float,
    null_threshold: float,
    candidate_temperature: float = 1.0,
    null_temperature: float = 1.0,
) -> dict[str, float]:
    model.eval()
    tp = fp = fn = 0
    neg_total = neg_correct = 0
    false_boxes_neg = 0
    sample_count = 0

    for raw in tqdm(loader, desc="Evaluation", leave=False):
        batch = move_batch(raw)
        out = model(
            batch["visual_features"], batch["boxes"], batch["detector_scores"],
            batch["candidate_mask"], batch["image_sizes"],
            batch["query_ids"], batch["query_mask"],
            batch["target_ids"], batch["target_text_mask"],
            batch["anchor_ids"], batch["anchor_text_mask"],
            batch["relation_ids"], batch["has_relation"],
        )
        cand_p = torch.sigmoid(out["candidate_logits"] / candidate_temperature)
        null_p = torch.sigmoid(out["null_logit"] / null_temperature)

        for i in range(len(raw["sample_id"])):
            valid = batch["candidate_mask"][i]
            choose_null = float(null_p[i]) >= null_threshold
            if choose_null:
                pred_boxes = batch["boxes"][i, :0]
            else:
                keep = valid & (cand_p[i] >= candidate_threshold)
                pred_boxes = batch["boxes"][i][keep]

            gt = raw["gt_boxes"][i]
            a, b, c = greedy_match_count(pred_boxes.detach().cpu(), gt, 0.5)
            tp += a; fp += b; fn += c
            sample_count += 1

            if len(gt) == 0:
                neg_total += 1
                if len(pred_boxes) == 0:
                    neg_correct += 1
                false_boxes_neg += len(pred_boxes)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "set_precision@0.5": precision,
        "set_recall@0.5": recall,
        "set_f1@0.5": f1,
        "no_target_accuracy": neg_correct / max(neg_total, 1),
        "false_boxes_per_negative": false_boxes_neg / max(neg_total, 1),
        "samples": sample_count,
    }

In [ ]:
model = DODVLM(cfg).to(DEVICE)

text_params = [p for p in model.text_encoder.parameters() if p.requires_grad]
other_params = [
    p for n, p in model.named_parameters()
    if not n.startswith("text_encoder.") and p.requires_grad
]

optimizer = torch.optim.AdamW(
    [
        {"params": other_params, "lr": cfg.lr_fusion},
        {"params": text_params, "lr": cfg.lr_text},
    ],
    weight_decay=cfg.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(cfg.epochs, 1),
)
scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp)

LAST_CKPT = CKPT_DIR / "last.pt"
BEST_CKPT = CKPT_DIR / "best.pt"
RECOVERY_CKPT = CKPT_DIR / "recovery.pt"

start_epoch = 0
resume_batch = 0
best_metric = -1.0
patience_left = cfg.early_stopping_patience

def load_training_checkpoint(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if ckpt.get("config_hash") != CONFIG_HASH:
        print(f"⚠ Bỏ qua {path.name}: config hash không khớp.")
        return None
    return ckpt

# Ưu tiên recovery vì nó có thể mới hơn last.pt.
resume_ckpt = load_training_checkpoint(RECOVERY_CKPT)
resume_source = RECOVERY_CKPT if resume_ckpt is not None else LAST_CKPT
if resume_ckpt is None:
    resume_ckpt = load_training_checkpoint(LAST_CKPT)

if resume_ckpt is not None:
    print("↻ Resume từ:", resume_source)
    model.load_state_dict(resume_ckpt["model"])
    optimizer.load_state_dict(resume_ckpt["optimizer"])
    scheduler.load_state_dict(resume_ckpt["scheduler"])
    scaler.load_state_dict(resume_ckpt["scaler"])
    best_metric = float(resume_ckpt.get("best_metric", -1.0))
    patience_left = int(
        resume_ckpt.get("patience_left", cfg.early_stopping_patience)
    )

    if resume_ckpt.get("kind") == "recovery":
        start_epoch = int(resume_ckpt["epoch"])
        resume_batch = int(resume_ckpt["batch_step"]) + 1
        print(f"  → tiếp tục epoch {start_epoch + 1}, batch {resume_batch}")
    else:
        start_epoch = int(resume_ckpt["epoch"]) + 1
        resume_batch = 0

history_path = RUN_DIR / "training_history.csv"
if history_path.exists():
    history = pd.read_csv(history_path).to_dict("records")
else:
    history: list[dict[str, Any]] = []

global_step = 0

def make_checkpoint(
    *,
    kind: str,
    epoch: int,
    batch_step: int,
    val_metrics: dict[str, float] | None = None,
) -> dict[str, Any]:
    return {
        "kind": kind,
        "config_hash": CONFIG_HASH,
        "cache_signature": CACHE_SIGNATURE,
        "epoch": epoch,
        "batch_step": batch_step,
        "best_metric": best_metric,
        "patience_left": patience_left,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "config": asdict(cfg),
        "val_metrics": val_metrics,
    }

with stage_timer("Session 7 — Training DOD-VLM"):
    for epoch in range(start_epoch, cfg.epochs):
        # Seed theo epoch => thứ tự shuffle tái lập được khi resume giữa epoch.
        train_loader = make_loader(
            "train",
            training=True,
            shuffle=True,
            seed=SEED + epoch,
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        seen_batches = 0
        epoch_start = time.perf_counter()
        last_recovery_save = time.perf_counter()

        pbar = tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Epoch {epoch+1:02d}/{cfg.epochs:02d}",
            unit="batch",
        )

        for step, raw in pbar:
            if epoch == start_epoch and step < resume_batch:
                continue

            batch = move_batch(raw)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=cfg.amp,
            ):
                out = model(
                    batch["visual_features"],
                    batch["boxes"],
                    batch["detector_scores"],
                    batch["candidate_mask"],
                    batch["image_sizes"],
                    batch["query_ids"],
                    batch["query_mask"],
                    batch["target_ids"],
                    batch["target_text_mask"],
                    batch["anchor_ids"],
                    batch["anchor_text_mask"],
                    batch["relation_ids"],
                    batch["has_relation"],
                )
                loss, loss_parts = compute_loss(out, batch)
                loss = loss / cfg.grad_accum_steps

            scaler.scale(loss).backward()

            should_step = (
                (step + 1) % cfg.grad_accum_steps == 0
                or (step + 1) == len(train_loader)
            )
            if should_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            global_step += 1
            seen_batches += 1
            running += float(loss.detach()) * cfg.grad_accum_steps

            pbar.set_postfix(
                loss=f"{running/max(seen_batches,1):.4f}",
                cand=f"{loss_parts['candidate']:.3f}",
                null=f"{loss_parts['null']:.3f}",
                inj=f"{float(batch['injected'].float().mean()):.2f}",
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            )

            # Recovery checkpoint định kỳ trong epoch.
            minutes_since_save = (
                time.perf_counter() - last_recovery_save
            ) / 60.0
            if minutes_since_save >= cfg.save_every_minutes:
                atomic_torch_save(
                    make_checkpoint(
                        kind="recovery",
                        epoch=epoch,
                        batch_step=step,
                    ),
                    RECOVERY_CKPT,
                )
                last_recovery_save = time.perf_counter()
                pbar.write(
                    f"💾 Recovery checkpoint @ epoch={epoch+1}, batch={step+1}"
                )

        # Epoch đã hoàn tất; recovery không còn cần.
        resume_batch = 0
        RECOVERY_CKPT.unlink(missing_ok=True)

        scheduler.step()

        val_metrics = evaluate_loader(
            model,
            val_loader,
            candidate_threshold=cfg.default_candidate_threshold,
            null_threshold=cfg.default_null_threshold,
        )
        metric = val_metrics["set_f1@0.5"]
        elapsed = time.perf_counter() - epoch_start

        row = {
            "epoch": epoch,
            "train_loss": running / max(seen_batches, 1),
            "minutes": elapsed / 60,
            **val_metrics,
        }
        history.append(row)
        print("\nValidation:", json.dumps(val_metrics, indent=2))
        print(f"Epoch time: {elapsed/60:.2f} phút")
        gpu_report("Training:")

        improved = metric > best_metric
        if improved:
            best_metric = metric
            patience_left = cfg.early_stopping_patience
        else:
            patience_left -= 1

        checkpoint = make_checkpoint(
            kind="epoch",
            epoch=epoch,
            batch_step=len(train_loader) - 1,
            val_metrics=val_metrics,
        )
        atomic_torch_save(checkpoint, LAST_CKPT)

        if improved:
            shutil.copy2(LAST_CKPT, BEST_CKPT)
            print(f"★ New best checkpoint: F1={best_metric:.4f}")

        pd.DataFrame(history).to_csv(history_path, index=False)

        if patience_left <= 0:
            print("Early stopping.")
            break

    state.mark(
        "training_complete",
        best_metric=best_metric,
        checkpoint=str(BEST_CKPT),
    )

## Session 8 — Calibration validation và đánh giá held-out

Không dùng threshold tùy ý cho no-target.

Notebook:
1. load `best.pt`;
2. cache logits trên validation;
3. grid search `candidate_threshold × null_threshold`;
4. chọn cặp threshold tối đa `set_f1@0.5`, với no-target accuracy làm tie-break;
5. khóa threshold;
6. đánh giá một lần trên test split.

In [ ]:
best = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(best["model"])
model.to(DEVICE).eval()

@torch.inference_mode()
def collect_eval_records(
    model: nn.Module,
    loader: DataLoader,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    model.eval()

    for raw in tqdm(loader, desc="Collect logits", unit="batch"):
        batch = move_batch(raw)
        out = model(
            batch["visual_features"],
            batch["boxes"],
            batch["detector_scores"],
            batch["candidate_mask"],
            batch["image_sizes"],
            batch["query_ids"],
            batch["query_mask"],
            batch["target_ids"],
            batch["target_text_mask"],
            batch["anchor_ids"],
            batch["anchor_text_mask"],
            batch["relation_ids"],
            batch["has_relation"],
        )

        cand_p = torch.sigmoid(out["candidate_logits"]).detach().cpu()
        null_p = torch.sigmoid(out["null_logit"]).detach().cpu()
        boxes = batch["boxes"].detach().cpu()
        masks = batch["candidate_mask"].detach().cpu()

        for i in range(len(raw["sample_id"])):
            n = int(masks[i].sum())
            records.append(
                {
                    "sample_id": raw["sample_id"][i],
                    "candidate_probs": cand_p[i, :n].clone(),
                    "boxes": boxes[i, :n].clone(),
                    "null_prob": float(null_p[i]),
                    "gt_boxes": raw["gt_boxes"][i].clone(),
                }
            )
    return records

def score_records(
    records: list[dict[str, Any]],
    candidate_threshold: float,
    null_threshold: float,
) -> dict[str, float]:
    tp = fp = fn = 0
    neg_total = neg_correct = 0
    false_boxes_neg = 0

    for rec in records:
        if rec["null_prob"] >= null_threshold:
            pred_boxes = rec["boxes"][:0]
        else:
            keep = rec["candidate_probs"] >= candidate_threshold
            pred_boxes = rec["boxes"][keep]

        gt = rec["gt_boxes"]
        a, b, c = greedy_match_count(pred_boxes, gt, 0.5)
        tp += a
        fp += b
        fn += c

        if len(gt) == 0:
            neg_total += 1
            if len(pred_boxes) == 0:
                neg_correct += 1
            false_boxes_neg += len(pred_boxes)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "set_precision@0.5": precision,
        "set_recall@0.5": recall,
        "set_f1@0.5": f1,
        "no_target_accuracy": neg_correct / max(neg_total, 1),
        "false_boxes_per_negative": false_boxes_neg / max(neg_total, 1),
        "samples": len(records),
    }

def calibrate_thresholds(
    records: list[dict[str, Any]],
) -> tuple[dict[str, float], pd.DataFrame]:
    grid = torch.linspace(
        0.15,
        0.85,
        cfg.threshold_grid_size,
    ).tolist()

    rows: list[dict[str, float]] = []
    best_cfg = {
        "candidate_threshold": cfg.default_candidate_threshold,
        "null_threshold": cfg.default_null_threshold,
        "score": -1.0,
        "no_target_accuracy": -1.0,
    }

    for ct in tqdm(grid, desc="Calibration candidate threshold", unit="thr"):
        for nt in grid:
            m = score_records(records, ct, nt)
            rows.append(
                {
                    "candidate_threshold": float(ct),
                    "null_threshold": float(nt),
                    **m,
                }
            )
            key = (m["set_f1@0.5"], m["no_target_accuracy"])
            old = (best_cfg["score"], best_cfg["no_target_accuracy"])
            if key > old:
                best_cfg = {
                    "candidate_threshold": float(ct),
                    "null_threshold": float(nt),
                    "score": float(m["set_f1@0.5"]),
                    "no_target_accuracy": float(m["no_target_accuracy"]),
                }

    return best_cfg, pd.DataFrame(rows)

with stage_timer("Session 8A — Cache validation logits"):
    val_records = collect_eval_records(model, val_loader)

with stage_timer("Session 8B — Calibration threshold"):
    calibration, calibration_table = calibrate_thresholds(val_records)
    atomic_json_dump(calibration, RUN_DIR / "calibration.json")
    calibration_table.to_csv(
        RUN_DIR / "calibration_grid.csv",
        index=False,
    )
    print(json.dumps(calibration, indent=2))

with stage_timer(f"Session 8C — Final evaluation trên split '{test_split}'"):
    test_records = collect_eval_records(model, test_loader)
    final_metrics = score_records(
        test_records,
        candidate_threshold=calibration["candidate_threshold"],
        null_threshold=calibration["null_threshold"],
    )
    atomic_json_dump(final_metrics, RUN_DIR / "final_metrics.json")
    print(json.dumps(final_metrics, indent=2))
    state.mark(
        "evaluation_complete",
        split=test_split,
        metrics=final_metrics,
    )

## Session 9 — Đóng gói final VLM, reload và inference end-to-end

Artifact cuối cùng là một **bundle đầy đủ**:
- trained DOD-VLM weights;
- tokenizer/text model identifier;
- proposal checkpoint name;
- visual encoder recipe;
- calibration;
- config;
- metric;
- cache/data metadata.

Sau khi save, notebook tạo **runtime mới từ disk**, load lại bundle và chạy inference trên một ảnh thật. Đây là kiểm tra cuối để chắc artifact không chỉ train được mà còn reload được.

In [ ]:
FINAL_BUNDLE = ARTIFACT_DIR / "dod_vlm_final.pt"

bundle = {
    "format_version": 1,
    "created_at": now_iso(),
    "config_hash": CONFIG_HASH,
    "cache_signature": CACHE_SIGNATURE,
    "config": asdict(cfg),
    "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "calibration": calibration,
    "final_metrics": final_metrics,
    "test_split": test_split,
    "architecture": {
        "proposal": "YOLOE-26s prompt-free",
        "visual_encoder": "FasterRCNN MobileNetV3-Large-FPN backbone + MultiScaleRoIAlign",
        "text_encoder": cfg.text_model,
        "fusion": "A0 generic object-text transformer + A1 structured relational branch + learned gate",
        "output": "independent candidate logits + explicit null head",
    },
}
atomic_torch_save(bundle, FINAL_BUNDLE)
atomic_json_dump(asdict(cfg), ARTIFACT_DIR / "config.json")
atomic_json_dump(final_metrics, ARTIFACT_DIR / "metrics.json")
atomic_json_dump(calibration, ARTIFACT_DIR / "calibration.json")

print("✓ Final bundle:", FINAL_BUNDLE)
print(f"Size: {FINAL_BUNDLE.stat().st_size / 2**20:.1f} MB")
state.mark("final_bundle_saved", path=str(FINAL_BUNDLE))

In [ ]:
class DODVLMRuntime:
    def __init__(self, bundle_path: Path, device: torch.device):
        self.bundle = torch.load(bundle_path, map_location="cpu", weights_only=False)
        self.cfg = Config(**self.bundle["config"])
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.text_model)
        self.model = DODVLM(self.cfg).to(device)
        self.model.load_state_dict(self.bundle["model_state"])
        self.model.eval()

        self.proposal = YOLOE(self.cfg.proposal_model)
        self.region_encoder = FrozenRegionEncoder(device)

        self.ct = float(self.bundle["calibration"]["candidate_threshold"])
        self.nt = float(self.bundle["calibration"]["null_threshold"])

    @torch.inference_mode()
    def predict(self, image: str | Path | Image.Image, query: str) -> list[dict[str, Any]]:
        if isinstance(image, (str, Path)):
            image_path = Path(image)
            pil = Image.open(image_path).convert("RGB")
            source = str(image_path)
        else:
            pil = image.convert("RGB")
            source = np.asarray(pil)

        res = self.proposal.predict(
            source=source,
            imgsz=self.cfg.proposal_imgsz,
            conf=self.cfg.proposal_conf,
            max_det=self.cfg.top_k,
            device=0,
            half=True,
            verbose=False,
        )[0]

        if res.boxes is None or len(res.boxes) == 0:
            return []

        boxes = res.boxes.xyxy.detach().cpu().float()[: self.cfg.top_k]
        scores = res.boxes.conf.detach().cpu().float()[: self.cfg.top_k]
        cls_ids = res.boxes.cls.detach().cpu().long().tolist()[: self.cfg.top_k]
        labels = [str(res.names[int(i)]) for i in cls_ids]
        feats = self.region_encoder.encode(pil, boxes)

        parsed = parse_query(query, target_categories=None)
        q = self.tokenizer([query], padding=True, truncation=True, max_length=self.cfg.max_text_len, return_tensors="pt")
        t = self.tokenizer([parsed["target_phrase"]], padding=True, truncation=True, max_length=16, return_tensors="pt")
        a = self.tokenizer([parsed["anchor_phrase"] or "[UNK]"], padding=True, truncation=True, max_length=16, return_tensors="pt")

        k = self.cfg.top_k
        n = min(k, len(boxes))
        vf = pad_2d(feats, k, self.cfg.visual_dim)[None].to(self.device)
        bx = pad_2d(boxes, k, 4)[None].to(self.device)
        sc = pad_1d(scores, k)[None].to(self.device)
        cm = torch.zeros((1, k), dtype=torch.bool, device=self.device)
        cm[:, :n] = True
        size = torch.tensor([[pil.height, pil.width]], dtype=torch.float32, device=self.device)

        out = self.model(
            vf, bx, sc, cm, size,
            q["input_ids"].to(self.device), q["attention_mask"].to(self.device),
            t["input_ids"].to(self.device), t["attention_mask"].to(self.device),
            a["input_ids"].to(self.device), a["attention_mask"].to(self.device),
            torch.tensor([parsed["relation_id"]], device=self.device),
            torch.tensor([parsed["relation_id"] != 0], device=self.device),
        )

        null_p = float(torch.sigmoid(out["null_logit"])[0])
        if null_p >= self.nt:
            return []

        cand_p = torch.sigmoid(out["candidate_logits"])[0, :n].cpu()
        keep = torch.where(cand_p >= self.ct)[0].tolist()

        return [
            {
                "box": [round(float(x), 2) for x in boxes[i].tolist()],
                "score": round(float(cand_p[i]), 4),
                "proposal_label": labels[i],
                "proposal_score": round(float(scores[i]), 4),
            }
            for i in keep
        ]

with stage_timer("Session 9 — Reload final bundle + inference sanity check"):
    # Giải phóng model train trước khi dựng runtime mới.
    del model
    gc.collect()
    torch.cuda.empty_cache()

    runtime = DODVLMRuntime(FINAL_BUNDLE, DEVICE)

    demo_row = manifest[manifest["split"] == test_split].iloc[0]
    demo_image = IMAGE_DIR / demo_row.file_name
    demo_query = str(demo_row.query)
    prediction = runtime.predict(demo_image, demo_query)

    print("Ảnh :", demo_image)
    print("Query:", demo_query)
    print("Prediction:", json.dumps(prediction, indent=2, ensure_ascii=False))
    state.mark("reload_inference_passed", sample_id=str(demo_row.sample_id))

# Hoàn tất

Nếu **Run All** đi tới đây không lỗi, project đã tạo:

```text
dod_vlm_workspace/
└── runs/
    └── dod_vlm_grefcoco_v1/
        ├── checkpoints/
        │   ├── last.pt
        │   └── best.pt
        ├── calibration.json
        ├── final_metrics.json
        ├── training_history.csv
        └── final_model/
            ├── dod_vlm_final.pt
            ├── config.json
            ├── calibration.json
            └── metrics.json
```

File cần giữ để triển khai là:

**`final_model/dod_vlm_final.pt`**

Runtime reference là class **`DODVLMRuntime`** trong Session 9.

## Những gì model cuối đã học

- grounding toàn câu qua A0;
- object-to-object interaction;
- target entity matching;
- latent anchor matching;
- relation-conditioned pairwise geometry;
- 0/1/N candidate scoring;
- explicit no-target rejection;
- query-dependent fusion giữa generic và structured reasoning.

Proposal/localization và ROI backbone được giữ frozen theo thiết kế nhằm giữ tính khả thi trên RTX A3000 12 GB và phục vụ mục tiêu real-time về sau.